# Step 3: Feature Engineering & Target Variable Creation

In this notebook, we transform our cleaned market data into machine learning features that capture:
- Market returns and momentum
- Lag features
- Technical Indicators (RSI, MACD, Moving Averages)
- Macroeconomic indicators (returns on Gold, Crude Oil, USD/INR, and changes in India VIX)
- Future Target variable (1 = Market goes UP next day, 0 = DOWN)

In [1]:
import pandas as pd
import numpy as np
from ta.momentum import RSIIndicator
from ta.trend import MACD

In [2]:
df = pd.read_csv("../data/processed/clean_market_data.csv")
df["Date"] = pd.to_datetime(df["Date"])
print(f"Loaded clean dataset shape: {df.shape}")
df.head()

Loaded clean dataset shape: (3928, 10)


,Date,NIFTY_Close,NIFTY_High,NIFTY_Low,NIFTY_Open,Volume,Gold,Oil,USD_INR,India_VIX
0,2010-01-04,5232.200195,5238.450195,5167.100098,5200.899902,0,1117.699951,81.510002,46.287998,23.639999
1,2010-01-05,5277.899902,5288.350098,5242.399902,5277.149902,0,1118.099976,81.769997,46.119999,22.270000
2,2010-01-06,5281.799805,5310.850098,5260.049805,5278.149902,0,1135.900024,83.180000,45.720001,22.120001
3,2010-01-07,5263.100098,5302.549805,5244.750000,5281.799805,0,1133.099976,82.660004,45.688000,22.500000
4,2010-01-08,5244.750000,5276.750000,5234.700195,5264.250000,0,1138.199951,82.750000,45.518002,22.570000


In [3]:
# Feature 1: NIFTY Daily Return
df["Return"] = df["NIFTY_Close"].pct_change()

# Feature 2: NIFTY Lag Returns
df["Lag_1"] = df["Return"].shift(1)
df["Lag_2"] = df["Return"].shift(2)
df["Lag_3"] = df["Return"].shift(3)

print("Returns and Lag Returns created.")

Returns and Lag Returns created.


In [4]:
# Feature 3: RSI (Relative Strength Index)
rsi = RSIIndicator(df["NIFTY_Close"], window=14)
df["RSI"] = rsi.rsi()

# Feature 4: MACD
macd = MACD(df["NIFTY_Close"])
df["MACD"] = macd.macd()
df["MACD_Signal"] = macd.macd_signal()
df["MACD_Hist"] = macd.macd_diff()

print("RSI and MACD indicators calculated.")

RSI and MACD indicators calculated.


In [5]:
# Feature 5 & 6: Moving Averages and Price Differences
df["MA20"] = df["NIFTY_Close"].rolling(20).mean()
df["MA50"] = df["NIFTY_Close"].rolling(50).mean()

df["Price_MA20_Diff"] = df["NIFTY_Close"] - df["MA20"]
df["Price_MA50_Diff"] = df["NIFTY_Close"] - df["MA50"]

print("Moving averages and MA differences calculated.")

Moving averages and MA differences calculated.


In [6]:
# Feature 7 & 8: Momentum and Volatility
df["Momentum_3"] = df["NIFTY_Close"] - df["NIFTY_Close"].shift(3)
df["Momentum_7"] = df["NIFTY_Close"] - df["NIFTY_Close"].shift(7)
df["Volatility"] = df["Return"].rolling(20).std()

print("Momentum and rolling Volatility calculated.")

Momentum and rolling Volatility calculated.


In [7]:
# Feature 9 - 12: Macroeconomic Returns/Changes
df["Gold_Return"] = df["Gold"].pct_change()
df["Oil_Return"] = df["Oil"].pct_change()
df["USD_Return"] = df["USD_INR"].pct_change()
df["VIX_Change"] = df["India_VIX"].pct_change()

print("Macroeconomic return features created.")

Macroeconomic return features created.


In [8]:
# Target Variable: 1 if future close > current close, 0 otherwise
df["Future_Close"] = df["NIFTY_Close"].shift(-1)
df["Target"] = (df["Future_Close"] > df["NIFTY_Close"]).astype(int)

print("Target variable created.")

Target variable created.


In [9]:
# Remove missing rows created by indicators/lags
print(f"Shape before dropping NaN rows: {df.shape}")
df = df.dropna().reset_index(drop=True)
print(f"Shape after dropping NaN rows: {df.shape}")

Shape before dropping NaN rows: (3928, 31)
Shape after dropping NaN rows: (3878, 31)


In [10]:
print("=== Columns in Final Dataset ===")
print(df.columns.tolist())
print("\n=== Target Class Distribution ===")
print(df["Target"].value_counts())
print(df["Target"].value_counts(normalize=True))

=== Columns in Final Dataset ===
['Date', 'NIFTY_Close', 'NIFTY_High', 'NIFTY_Low', 'NIFTY_Open', 'Volume', 'Gold', 'Oil', 'USD_INR', 'India_VIX', 'Return', 'Lag_1', 'Lag_2', 'Lag_3', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'MA20', 'MA50', 'Price_MA20_Diff', 'Price_MA50_Diff', 'Momentum_3', 'Momentum_7', 'Volatility', 'Gold_Return', 'Oil_Return', 'USD_Return', 'VIX_Change', 'Future_Close', 'Target']

=== Target Class Distribution ===
Target
1    2061
0    1817
Name: count, dtype: int64
Target
1    0.53146
0    0.46854
Name: proportion, dtype: float64


In [11]:
# Save dataset
df.to_csv("../data/processed/feature_engineered_data.csv", index=False)
print("Saved feature engineered dataset to data/processed/feature_engineered_data.csv")

Saved feature engineered dataset to data/processed/feature_engineered_data.csv
